# Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score


# Load Dataset

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
TRAIN_PATH = '/content/drive/MyDrive/Datasets/HustleFlow/train_raw.csv'
TEST_PATH = '/content/drive/MyDrive/Datasets/HustleFlow/test_raw.csv'

In [4]:
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)
print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

Train shape: (960, 28)
Test shape: (240, 28)


# Overview

In [5]:
TARGET_COL = 'PerformanceRating'
ID_COL = 'EmpNumber'

# Check distribution
print("\n--- Target Distribution (Train set) ---")
distribution = df_train[TARGET_COL].value_counts(normalize=True).sort_index() * 100
print(distribution)


--- Target Distribution (Train set) ---
PerformanceRating
0    16.145833
1    72.812500
2    11.041667
Name: proportion, dtype: float64


# Preprocessing

In [6]:
# Preprocess
X_train = df_train.drop(columns=[TARGET_COL, ID_COL])
y_train = df_train[TARGET_COL]

X_test = df_test.drop(columns=[TARGET_COL, ID_COL])
y_test_actual = df_test[TARGET_COL]


# Automatically identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

# Model Pipeline

In [7]:
# Optional (make sure for work afterthat)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

# Preprocessing for categorical data: Impute with constant & One-Hot Encode
# handle_unknown='ignore' prevents errors if test data has new categories
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Initialize Random Forest Model
# class_weight='balanced': Adjusts weights inversely proportional to class frequencies
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# Create the full pipeline: Preprocessing -> Model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf)
])


# Hyperparameter Tuning

In [8]:
param_dist = {
    'model__n_estimators': [100, 200, 300, 500],        # Number of trees
    'model__max_depth': [10, 15, 20, 30, None],         # Maximum depth of tree
    'model__min_samples_split': [2, 5, 10],             # Min samples required to split a node
    'model__min_samples_leaf': [1, 2, 4],               # Min samples required at a leaf node
    'model__max_features': ['sqrt', 'log2']             # Number of features to consider at best split
}

print("\nStarting RandomizedSearchCV (Optimizing for F1-Macro)...")

# Setup Randomized Search
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,                       # Number of parameter settings that are sampled
    cv=StratifiedKFold(n_splits=5),  # Stratified K-Fold to preserve class percentage
    verbose=1,                       # Controls the verbosity: the higher, the more messages
    random_state=42,
    n_jobs=-1,                       # Use all processors
    scoring='f1_macro'
)


random_search.fit(X_train, y_train)






Starting RandomizedSearchCV (Optimizing for F1-Macro)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer())]),
                                                                               Index(['Age', 'DistanceFromHome', 'EmpEducationLevel',
       'EmpEnvironmentSatisfaction', 'EmpHourlyRate', 'EmpJobInvolvement',
       'EmpJobLevel', 'EmpJobSat...
      dtype='object'))])),
                                             ('model',
                                              RandomForestClassifier(class_weight='balanced',
                                                                     random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__max_depth': [10, 15, 20, 30,
                                                             None],
                                        'model__max_features': ['sqrt', 'log2'],
                                        'model__min_samples_leaf': [1, 2, 4],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [100, 200, 300,
                                                                500]},
                   random_state=42, scoring='f1_macro', verbose=1)

In [9]:
print(f"\nBest F1-Macro Score (CV): {random_search.best_score_:.4f}")
print(f"Best Parameters: {random_search.best_params_}")


Best F1-Macro Score (CV): 0.8781
Best Parameters: {'model__n_estimators': 300, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 15}


In [10]:
from sklearn.model_selection import cross_validate, StratifiedKFold
import pandas as pd

# Best model
best_model = random_search.best_estimator_

# Metrics
scoring_metrics = {
    'accuracy': 'accuracy',
    'f1_macro': 'f1_macro',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro'
}

# Cross-validate
cv_full = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=scoring_metrics,
    return_train_score=False
)

# Tạo DataFrame + rename cho đẹp
df_cv = pd.DataFrame(cv_full).rename(columns={
    'fit_time': 'Fit Time (s)',
    'score_time': 'Score Time (s)',
    'test_accuracy': 'Accuracy',
    'test_f1_macro': 'F1-Macro',
    'test_precision_macro': 'Precision',
    'test_recall_macro': 'Recall'
})

# Đặt tên index
df_cv.index = [f'Fold {i+1}' for i in range(len(df_cv))]

# Làm tròn
df_cv = df_cv.round(4)

# Thêm Average & Std
df_cv.loc['Average'] = df_cv.mean()
df_cv.loc['Std Dev'] = df_cv.std()

print("\n CROSS-VALIDATION PERFORMANCE (5-FOLD)\n")

# Style cho đẹp
styled_df = (
    df_cv.style
        .format("{:.4f}")
        .set_properties(**{
            'text-align': 'center',
            'font-size': '13px'
        })
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '13px')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '14px')]}
        ])
        .background_gradient(
            cmap='Blues',
            subset=['Accuracy', 'F1-Macro', 'Precision', 'Recall']
        )
)

display(styled_df)



 CROSS-VALIDATION PERFORMANCE (5-FOLD)



,Fit Time (s),Score Time (s),Accuracy,F1-Macro,Precision,Recall
Fold 1,0.9137,0.0526,0.9271,0.8838,0.9407,0.8439
Fold 2,0.8741,0.0558,0.9219,0.8695,0.8999,0.8498
Fold 3,0.8708,0.0373,0.9115,0.8470,0.8819,0.8265
Fold 4,0.5861,0.0365,0.9375,0.8936,0.9423,0.8686
Fold 5,0.5905,0.0372,0.9375,0.9080,0.9242,0.8951
Average,0.7670,0.0439,0.9271,0.8804,0.9178,0.8568
Std Dev,0.1467,0.0085,0.0099,0.0209,0.0236,0.0234


In [11]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Predict
y_pred = random_search.predict(X_test)

# Metrics chính
test_acc = accuracy_score(y_test_actual, y_pred)
test_f1 = f1_score(y_test_actual, y_pred, average='macro')

df_test_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Macro'],
    'Score': [test_acc, test_f1]
}).set_index('Metric').round(4)

print("\n TEST SET PERFORMANCE SUMMARY\n")
display(
    df_test_summary.style
        .format("{:.4f}")
        .set_properties(**{'text-align': 'center', 'font-size': '13px'})
)

# Classification Report -> DataFrame
df_report = pd.DataFrame(
    classification_report(
        y_test_actual,
        y_pred,
        output_dict=True
    )
).transpose().round(4)

print("\n DETAILED CLASSIFICATION REPORT\n")

display(
    df_report.style
        .background_gradient(
            cmap='Greens',
            subset=['precision', 'recall', 'f1-score']
        )
        .set_properties(**{'text-align': 'center', 'font-size': '12px'})
)



 TEST SET PERFORMANCE SUMMARY



,Score
Metric,
Accuracy,0.9292
F1-Macro,0.8862



 DETAILED CLASSIFICATION REPORT



,precision,recall,f1-score,support
0,0.916700,0.846200,0.880000,39.000000
1,0.929300,0.977100,0.952600,175.000000
2,0.950000,0.730800,0.826100,26.000000
accuracy,0.929200,0.929200,0.929200,0.929200
macro avg,0.932000,0.851400,0.886200,240.000000
weighted avg,0.929500,0.929200,0.927100,240.000000


In [ ]:
import joblib
# Lưu model xuống file
joblib.dump(random_search.best_estimator_, 'model.pkl')

from google.colab import files
files.download('model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>